# 06 — Fine-tuning de CAMeLBERT et témoin CLS gelé

Nous entraînons les poids de l'encodeur, en plus de la tête finale. Le notebook05 utilisait un encodeur gelé et un résumé moyen ; ici nous utilisons la tête BERT native CLS/pooler prévue au cahier. Un **nouveau témoin CLS gelé**, construit avec la même tête, permet de contrôler la comparaison. La référence moyenne précédente reste conservée.

**L05, L07, L12 partiels.** Protocole : `docs/46_protocole_fine_tuning_encodeur.md`. Train/validation v2 ; aucun test officiel dans les scores. Le notebook délègue les calculs à `environments/encoder/`, depuis le même noyau **TuniScope (notebooks)**.

## Avant de lancer

Les scripts, sources autorisées et poids généraux épinglés doivent être présents. Aucun téléchargement implicite. **FULL est le mode par défaut** : un smoke apparié, puis tous les exemples et trois graines. Le régime `paired` produit douze modèles (deux tâches × trois graines × deux régimes). Le régime `finetuned` en produit six adaptés seulement. L'estimation locale depuis le premier essai est d'environ une heure pour les adaptés, 1 h 15 pour la comparaison appariée, sans garantie de durée ; les poids prennent plusieurs Go.

`smoke` s'arrête au contrôle sur100 lignes par partition/tâche. `replay` ne réentraîne pas et exige un mode de run explicitement indiqué. `off` ne charge rien. Le smoke MPS utilise `warn_only` pour les opérations sans implémentation déterministe ; graines fixées ne signifie pas répétabilité bit à bit garantie.


In [1]:
from pathlib import Path
import os
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'scripts/encoder_uv.sh').is_file()), None)
if ROOT is None:
    raise RuntimeError('Ouvrir depuis le dépôt ou son dossier notebooks/.')
sys.path.insert(0, str(ROOT / 'scripts'))
from encoder_finetuning_notebook_support import notebook_run, figures

TRAINING_MODE = os.environ.get('TUNISCOPE_FINETUNING_MODE', 'replay')
REGIME = os.environ.get('TUNISCOPE_FINETUNING_REGIME', 'paired')
REPLAY_RUN = os.environ.get('TUNISCOPE_FINETUNING_RUN',
    'outputs/encoder_finetuning/2026-11-05_cls_paired_mps_full')
REPLAY_EXPECTED_MODE = os.environ.get('TUNISCOPE_FINETUNING_REPLAY_MODE', 'full')
if TRAINING_MODE not in {'full', 'smoke', 'replay', 'off'}:
    raise ValueError('Choisir full, smoke, replay ou off.')
if REPLAY_EXPECTED_MODE not in {'full', 'smoke'}:
    raise ValueError('Le mode attendu doit être explicite : full ou smoke.')
print('Mode :', TRAINING_MODE.upper(), '| régime :', REGIME)


Mode : REPLAY | régime : paired


## 1. Ce qui change pendant l'apprentissage

`frozen` : seuls les1538/2307 paramètres de la tête changent ; le backbone avec pooler reste identique.

`finetuned` : les109082882/109083651 paramètres du classifieur complet sont entraînables. Nous vérifions aussi les gradients et le changement d'une matrice d'attention du premier bloc Transformer, pour ne pas confondre adaptation du pooler et adaptation du Transformer.

Les deux régimes ont la même architecture, initialisation par graine, ordre des données et optimiseur. Le dropout du backbone est désactivé dans les deux ; celui de la tête (0,1) reste actif pendant l'apprentissage. **`eval()` ne bloque pas les gradients** : le gel est contrôlé séparément avec `requires_grad`.

Batch physique8, accumulation2 : les gradients de deux petits lots contribuent à une mise à jour, soit16 textes effectifs. AdamW2e-5, weight decay0,01, maximum3 époques, patience2. Chaque époque lit tout le train du mode choisi ; la validation n'est jamais utilisée pour mettre à jour les poids.

Le témoin CLS à budget identique n'est pas la recherche du meilleur modèle gelé possible. Nous gardons aussi le témoin mean-pooling/tête du notebook05, entraîné selon un protocole distinct, comme référence complémentaire.


In [3]:
run_directory = None
run_result = None
if TRAINING_MODE != 'off':
    run_directory, run_result = notebook_run(TRAINING_MODE, REGIME, REPLAY_RUN, REPLAY_EXPECTED_MODE)
    print('\nDossier du run :', run_directory)
else:
    print('Mode OFF : aucun modèle ni corpus chargé.')


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at /Users/elghoth/Desktop/NLP Project/.tools/encoders/camelbert_mix_9be352797bdf28a9ae21e2ae582aaaca7abdb22d and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at /Users/elghoth/Desktop/NLP Project/.tools/encoders/camelbert_mix_9be352

## 2. Lire les métriques et distinguer l'essai du résultat complet

Accuracy = proportion de labels corrects. Macro-F1 = moyenne des F1 des classes, notre métrique principale. La meilleure époque est sélectionnée sur la validation ; toutes les graines prévues sont rapportées.

Un SMOKE porte sur100 textes par partition et tâche : ses scores ne sont pas comparables aux résultats FULL. Le vérificateur recharge tous les modèles sélectionnés et rejoue toute la validation du mode, avec contrôle des poids, identifiants, labels et probabilités. Les dix premiers exemples par tâche font aussi l'objet d'un contrôle explicite de tolérance numérique.


In [ ]:
if run_result is not None:
    print('Mode réellement exécuté :', run_result['mode'].upper())
    print('Corpus  Régime       Graine   Train   Val    Accuracy   Macro-F1   Époque')
    for r in run_result['results']:
        print(f"{r['corpus'].upper():<8}{r['regime']:<13}{r['seed']:<9}"
              f"{r['train']['rows']:<8}{r['validation']['rows']:<7}"
              f"{r['metrics']['accuracy']:<11.6f}{r['metrics']['macro_f1']:<11.6f}{r['best_epoch']}")
    for name, item in run_result['aggregates'].items():
        spread = item['sample_std_macro_f1']
        suffix = f" ± {spread:.6f}" if spread is not None else ' (une graine, essai technique)'
        print(f"{name} macro-F1 : {item['mean_macro_f1']:.6f}{suffix}")
    print('Appareil :', run_result['device'], '| test officiel utilisé :', run_result['test_loaded'])
    print(f"Durée réelle du run : {run_result['seconds'] / 60:.1f} min")


## 3. Images et fichiers sauvegardés

Chaque figure montre les pertes, le macro-F1 de validation et la matrice de confusion du checkpoint retenu. Les images portent explicitement le mode smoke/full et sont exportées depuis le même run. Les lignes de la matrice sont les labels réels ; ses colonnes sont les prédictions.

Dans chaque sous-dossier modèle : `history.json`, `predictions.json`, `config.json`, mappings/tokenizer et `tuniscope.json`. Les modèles adaptés contiennent `model.safetensors` ; les témoins contiennent `head.pt` et réutilisent la base exacte. Le dossier parent contient `results.json`, `summary.txt`, `verification.json` et `figures/`. Tout reste privé sous `outputs/`.


In [ ]:
if run_directory is not None:
    from IPython.display import display, Image
    paths = figures(run_directory)
    print('Images enregistrées dans :', run_directory / 'figures')
    for path in paths:
        display(Image(filename=str(path)))


## Après ce notebook

Les résultats de validation ne constituent pas une évaluation finale. Il reste à comparer les systèmes, examiner les erreurs, finaliser les cartes et les configurations retenues, puis évaluer le test selon le protocole final. Les modèles de l'application ne sont pas remplacés automatiquement. Aucun LLM ni adaptateur LoRA n'est entraîné ici.
